In [ ]:
import osmnx as ox
from osmnx.features import features_from_bbox
import networkx as nx
import geopandas as gpd
from src.geometric_utils import *
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from fancyimpute import SoftImpute

In [ ]:
df = pd.read_parquet('data/processed_data/S1CO2-approx-coordinates.parquet')

In [ ]:
# 1) define your area of interest in lon/lat
west, south, east, north = 9.2257, 45.47162, 9.23768, 45.48537
bbox = (west, south, east, north)

# ────────────────
# 2) street network (all modes: drive+walk+bike…)
# ────────────────
G = ox.graph.graph_from_bbox(
    bbox=bbox,
    network_type="all",             # drive, walk, bike, all_public, etc.
)  # multiDiGraph of every way in bbox :contentReference[oaicite:0]{index=0}
nodes, edges = ox.graph_to_gdfs(G)

# ────────────────
# 3) vector features via Overpass
# ────────────────
# a) building footprints (+ any height/levels tags)
tags_buildings = {"building": True}
gdf_buildings = features_from_bbox(bbox, tags_buildings)
# returns a GeoDataFrame multi-indexed by element type & OSM ID :contentReference[oaicite:1]{index=1}

# b) parks & town squares
tags_parks = {
    "leisure": ["park", "garden"],
    "amenity": ["town_square"]
}
gdf_parks = features_from_bbox(bbox, tags_parks)

# c) land-use areas (residential, commercial, industrial…)
tags_landuse = {"landuse": True}
gdf_landuse = features_from_bbox(bbox, tags_landuse)

# d) water bodies & waterways
tags_water = {"natural": ["water", "wetland"], "waterway": True}
gdf_water = features_from_bbox(bbox, tags_water)

# e) points of interest (amenities, shops, tourism…)
tags_pois = {"amenity": True, "shop": True, "tourism": True}
gdf_pois = features_from_bbox(bbox, tags_pois)

In [ ]:
# ────────────────
# 4) quick inspection / plotting
# ────────────────

print("Street segments:", len(edges))
print("Buildings:", len(gdf_buildings))
print("Parks/squares:", len(gdf_parks))

# example plot
ax = edges.plot(linewidth=0.5, color="gray", figsize=(10, 10))
gdf_parks.plot(ax=ax, facecolor="green", alpha=0.4)
gdf_buildings.plot(ax=ax, facecolor="lightgray", edgecolor="black", alpha=0.7)
nodes.plot(ax=ax, color="blue", markersize=5)

# plot df.x and df.y in the same plot
ax.scatter(df.x, df.y, color='red', s=10, label='Sample Points')
ax.set_title("OSM Data with Sample Points")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.show()


In [ ]:
nodes

In [ ]:
# 3) select the columns to impute (drop IDs, names, geometry, etc.)
exclude = ['osmid', 'name', 'ref', 'u', 'v', 'key', 'geometry']
use_cols = [c for c in edges.columns if c not in exclude]

# 4) detect list-valued columns and collapse each list → its 0th element
list_cols = [
    c for c in use_cols
    if edges[c].apply(lambda v: isinstance(v, list)).any()
]
X = edges[use_cols].copy()
for col in list_cols:
    X[col] = X[col].apply(lambda v: v[0] if isinstance(v, list) and v else v)

# ──────────────────────────────────────────────
# 5) coerce any “numeric‐looking” object columns → actual floats
for col in X.columns:
    if X[col].dtype == object:
        coerced = pd.to_numeric(X[col], errors='coerce')
        # if more than half of the entries became numeric, keep the coercion
        if coerced.notna().sum() > len(coerced) / 2:
            X[col] = coerced

# 6) inspect which columns truly have missing values
print("Missing before impute:")
print(X.isna().sum()[X.isna().sum() > 0])

# ──────────────────────────────────────────────
# 7) split into numeric vs. categorical, one-hot encode categoricals
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

dummies = pd.get_dummies(X[cat_cols], dummy_na=False)
M = pd.concat([X[num_cols], dummies], axis=1).astype(float)

# ──────────────────────────────────────────────
# 8) run SoftImpute (low-rank SVD imputation)
#    install fancyimpute: pip install fancyimpute
filled_array = SoftImpute(max_rank=20, max_iters=100).fit_transform(M.values)
M_filled = pd.DataFrame(filled_array, columns=M.columns, index=M.index)

# ──────────────────────────────────────────────
# 9) map the imputed values back into a new GeoDataFrame
edges_imputed = edges.copy()

# 9a) numeric columns
for col in num_cols:
    edges_imputed[col] = M_filled[col]

# 9b) categorical columns: pick the dummy with highest imputed score
for col in cat_cols:
    prefix    = f"{col}_"
    dcols     = [c for c in M_filled.columns if c.startswith(prefix)]
    if not dcols:
        continue
    best = M_filled[dcols].idxmax(axis=1).str[len(prefix):]
    edges_imputed[col] = best.astype(edges[col].dtype)

# 9c) restore geometry
edges_imputed.geometry = edges.geometry

# ──────────────────────────────────────────────
# 10) check remaining missingness
print("Missing after impute:")
print(edges_imputed[use_cols].isna().sum()[edges_imputed[use_cols].isna().sum() > 0])

In [ ]:
edges_imputed

In [ ]:
# among the columns you plan to impute
missing_counts = X.isna().sum()
print("Columns with missing values:\n", missing_counts[missing_counts>0])
print("Would impute these columns:", use_cols)

In [ ]:
for col in use_cols:
    if X[col].dtype == object and X[col].str.isnumeric().any():
        X[col] = pd.to_numeric(X[col], errors='coerce')


In [ ]:
edges['lanes'] = edges['lanes'].fillna(0)
edges['oneway'] = edges['oneway'].astype(int)
edges[edges['reversed'].apply(lambda x:str(x)) == str([False, True])]